# K-MEM-MEAS-2 Campaign Runner — Patch 2984

Launcher/monitor for the ensemble measurement campaign (frozen preregistration:
`kmem_meas2_ensemble_prereg.md`, Patch 2981; instrument: Patch 2983). This notebook is a
**convenience wrapper only** — the physics runs in the committed driver
`code/2983_kmem2_driver.py` on the committed 2902 CPU engine, exactly as frozen. The
notebook launches it as a subprocess, shows live progress, and helps with the final
commit-and-push.

**GPU note (why this stays on CPU):** the frozen preregistration binds the measurement to
the *committed 2902 engine, unmodified* — any physics-touching change voids the run. A GPU
port is a re-implementation of the physics, and because the Sea is deterministically
chaotic (the registered 2908 finding), GPU floating-point differences would diverge
trajectories, so "equivalence" could not be verified after the fact. The GPU therefore
cannot be used for **this** measurement without voiding the prereg. A validated GPU engine
is a legitimate *future* instrument: it would need its own equivalence-validation protocol
and would serve future preregistrations. Registered as a future-work note in this patch.

**Practical:** run this in Anaconda (base env is fine — only numpy is needed). Keep the
machine awake (Windows: Settings → System → Power → set Sleep to Never while the campaign
runs). The campaign is stop/restart safe at any time — finished legs are never redone.

In [ ]:
# Cell 1 — locate the repo and the driver; show campaign status
import os, sys, json, glob, subprocess
from pathlib import Path

CANDIDATES = [
    Path.home() / "Documents/GitHub/CPP",            # ClearPC
    Path.home() / "OneDrive/Documents/GitHub/CPP",   # Surface
    Path.cwd(),
]
REPO = next((p for p in CANDIDATES if (p / "series_phenomena").exists()), None)
assert REPO is not None, "CPP repo not found — edit CANDIDATES with your repo path."
DM = REPO / "series_phenomena/cosmology/dark_matter"
DRIVER = DM / "code/2983_kmem2_driver.py"
DATA = DM / "data/kmem2"
assert DRIVER.exists(), f"driver missing at {DRIVER} — pull latest main first."

TOTAL_LEGS = 2*512 + 2*128   # 1280, per the frozen prereg
done = len(glob.glob(str(DATA / "leg_*.json"))) if DATA.exists() else 0
print(f"repo:   {REPO}")
print(f"driver: {DRIVER.name}  |  python: {sys.version.split()[0]}")
print(f"campaign status: {done}/{TOTAL_LEGS} legs complete")

In [ ]:
# Cell 2 — smoke test (safe, ~1 minute, touches nothing evidentiary)
r = subprocess.run([sys.executable, str(DRIVER), "--smoke"],
                   cwd=str(DM), capture_output=True, text=True)
print(r.stdout, r.stderr)
assert "bit-identity for same seed: True" in r.stdout, "smoke failed — stop and report."
print("Smoke OK — safe to launch the campaign.")

In [ ]:
# Cell 3 — LAUNCH THE CAMPAIGN (long-running; safe to interrupt and re-run)
WORKERS = 12          # set to (your physical cores - 1) or so; higher = faster
MAX_LEGS = 0          # 0 = run to completion; set e.g. 60 for a bounded chunk

cmd = [sys.executable, str(DRIVER), "--workers", str(WORKERS)]
if MAX_LEGS: cmd += ["--max-legs", str(MAX_LEGS)]
print("launching:", " ".join(cmd)); print("-"*70)
proc = subprocess.Popen(cmd, cwd=str(DM), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    proc.terminate()
    print("\n[interrupted — progress is saved; re-run this cell anytime to resume]")
proc.wait()
print("-"*70); print("cell finished with code", proc.returncode)

In [ ]:
# Cell 4 — progress and rough ETA (run anytime, even while Cell 3 runs elsewhere)
import time
files = sorted(glob.glob(str(DATA / "leg_*.json"))) if DATA.exists() else []
done = len(files)
walls = []
for f in files[-40:]:
    try: walls.append(json.load(open(f)).get("wall_s", 0))
    except Exception: pass
avg = (sum(walls)/len(walls)) if walls else 16*60
remaining = TOTAL_LEGS - done
print(f"{done}/{TOTAL_LEGS} legs complete ({100*done/TOTAL_LEGS:.1f}%)")
print(f"recent avg leg wall: {avg/60:.1f} min  ->  ETA at {12} workers: "
      f"~{remaining*avg/12/3600:.1f} h remaining")

In [ ]:
# Cell 5 — after CAMPAIGN COMPLETE: commit and push the data
RUN_GIT = False   # set True to execute; False just prints the commands
cmds = [
    ["git", "add", "series_phenomena/cosmology/dark_matter/data/kmem2"],
    ["git", "commit", "-m",
     "MEAS-2 campaign data: all 1280 legs (founder execution per Patch 2983)"],
    ["git", "push", "origin", "main"],
]
done = len(glob.glob(str(DATA / "leg_*.json")))
if done < TOTAL_LEGS:
    print(f"NOT COMPLETE yet ({done}/{TOTAL_LEGS}) — finish the campaign first.")
else:
    for c in cmds:
        print("$", " ".join(c))
        if RUN_GIT:
            r = subprocess.run(c, cwd=str(REPO), capture_output=True, text=True)
            print(r.stdout, r.stderr)
    if not RUN_GIT:
        print("\n(set RUN_GIT = True and re-run to execute, or paste these in Git Bash)")